# Automating the Day 1 Wikipedia API Workflow with Cron

This notebook is a short setup guide for running the Day 1 Wikipedia API workflow from the command line and scheduling it with `crontab -e`.

The important idea:

- the **Python script** contains the collection logic;
- **cron** only decides when to run the script;
- each run writes a new folder with config, raw data, processed tables, logs, monitoring reports, version info, and a manifest;
- the workflow also keeps a small **continuation state** file so the next scheduled run can request the next Wikipedia search results instead of repeating the same first results.

## 1. Start in the Project Folder

Open Terminal and move into the course repository:

```bash
cd /Users/noelle/Documents/methodsNET
```

All commands below assume that this is the current working directory.


In [ ]:
# This cell shows the command, but does not run it.
print("cd /Users/noelle/Documents/methodsNET")


## 2. Run the Workflow Once Manually

Before scheduling anything, run the workflow manually. This checks that the script works, the environment is correct, and the output folders are created.


In [ ]:
manual_command = """python scripts/runnable_workflows/01_api_wikipedia.py \
  --query "digital services act" \
  --pages 1 \
  --page-size 2 \
  --extract-chars 300 \
  --run-label day1_api_test \
  --outdir data/runs \
  --state-dir data/state"""

print(manual_command)

The command should print paths similar to:

```text
Run folder: data/runs/20260702T..._day1_api_test
Raw JSONL: ...
Processed CSV: ...
Log: ...
Monitoring: ...
Manifest: ...
Monitoring passed: True
```


## 3. Inspect the New Run Folder

List the newest run folders:

```bash
ls -lt data/runs
```

Then replace `YOUR_RUN_FOLDER_NAME` with the newest folder name and inspect the files:

```bash
find data/runs/YOUR_RUN_FOLDER_NAME -maxdepth 3 -type f
```


In [ ]:
print("ls -lt data/runs")
print("find data/runs/YOUR_RUN_FOLDER_NAME -maxdepth 3 -type f")


A successful run should contain:

```text
config/config_snapshot.json
raw/wikipedia_api_raw.jsonl
processed/wikipedia_api_results.csv
logs/run.log
reports/monitoring_report.json
reports/monitoring_summary.md
reports/version_info.json
reports/scheduling_examples.md
reports/manifest.json
```


## 4. Check the Processed Table

The processed CSV is the analysis-friendly output:

```bash
head data/runs/YOUR_RUN_FOLDER_NAME/processed/wikipedia_api_results.csv
```


In [ ]:
print("head data/runs/YOUR_RUN_FOLDER_NAME/processed/wikipedia_api_results.csv")


## 5. Check the Log

The log records what happened while the script ran:

```bash
cat data/runs/YOUR_RUN_FOLDER_NAME/logs/run.log
```

This is useful because scheduled jobs run when nobody is watching the terminal.


In [ ]:
print("cat data/runs/YOUR_RUN_FOLDER_NAME/logs/run.log")


## 6. Check Monitoring

The monitoring summary gives a quick pass/fail result:

```bash
cat data/runs/YOUR_RUN_FOLDER_NAME/reports/monitoring_summary.md
```

Look for:

```text
Monitoring Summary: PASS
```


In [ ]:
print("cat data/runs/YOUR_RUN_FOLDER_NAME/reports/monitoring_summary.md")


## 7. Find the Python Path

Cron may not use the same Python environment as your normal Terminal.

Run:

```bash
which python
```

If it prints something like:

```text
/opt/anaconda3/bin/python
```

use that full path in the cron command.


In [ ]:
print("which python")


## 8. Prepare a Cron Log File

The scheduled job should append terminal output and errors to a log file:

```bash
mkdir -p data
touch data/wikipedia_api_cron.log
```


In [ ]:
print("mkdir -p data")
print("touch data/wikipedia_api_cron.log")


## 9. Open Crontab

Open the cron editor:

```bash
crontab -e
```

If the system asks you to choose an editor, choose the simplest option if unsure.


In [ ]:
print("crontab -e")


## 10. Add a Scheduled Job

Example: run every Monday at 09:00.

If `which python` returned `/opt/anaconda3/bin/python3`, use this:

```cron
0 9 * * 1 cd /Users/noelle/Documents/methodsNET && /opt/anaconda3/bin/python3 scripts/runnable_workflows/01_api_wikipedia.py --query "digital services act" --pages 1 --page-size 10 --extract-chars 500 --run-label scheduled_wikipedia_api --outdir data/runs --state-dir data/state >> data/wikipedia_api_cron.log 2>&1
```

The `--state-dir data/state` part stores the continuation token from the Wikipedia API. That is what makes the next scheduled run move forward to the next search results.

In [ ]:
cron_command = """0 9 * * 1 cd /Users/noelle/Documents/methodsNET && /opt/anaconda3/bin/python3 scripts/runnable_workflows/01_api_wikipedia.py --query "digital services act" --pages 1 --page-size 10 --extract-chars 500 --run-label scheduled_wikipedia_api --outdir data/runs --state-dir data/state >> data/wikipedia_api_cron.log 2>&1"""

print(cron_command)

## 11. Save and Exit Crontab

Depending on the editor:

- `nano`: press `Ctrl+O`, Enter, then `Ctrl+X`
- `vim`: press `Esc`, type `:wq`, press Enter


## 12. Confirm the Cron Job Was Saved

After saving, list current cron jobs:

```bash
crontab -l
```

You should see the scheduled line.


In [ ]:
print("crontab -l")


## 13. Test with a Near-Time Schedule

For a live demo, schedule the command for the next minute or two.

Cron format is:

```text
minute hour day-of-month month day-of-week command
```

Example: if it is currently 10:14, test with 10:16:

```cron
16 10 * * * cd /Users/noelle/Documents/methodsNET && /opt/anaconda3/bin/python3 scripts/runnable_workflows/01_api_wikipedia.py --query "digital services act" --pages 1 --page-size 2 --extract-chars 300 --run-label cron_test --outdir data/runs --state-dir data/state >> data/wikipedia_api_cron.log 2>&1
```

After the time passes, check:

```bash
ls -lt data/runs
cat data/wikipedia_api_cron.log
ls -lt data/state
```

The run folders appear under `data/runs/`. The continuation file appears under `data/state/`.

In [ ]:
print("ls -lt data/runs")
print("cat data/wikipedia_api_cron.log")
print("ls -lt data/state")

## 14. Remove the Test Schedule

Open crontab again:

```bash
crontab -e
```

Delete the temporary test line and save.


In [ ]:
print("crontab -e")


# Key Takeaway

Cron does **not** contain the collection logic.

Cron only says:

```text
when to run it
where to run it from
where terminal output/errors should go
```

The Python script does the real workflow:

```text
collect data
write raw evidence
write processed CSV
write logs
run monitoring checks
write manifest
write version info
save continuation state for the next scheduled run
```

If you want to start again from the first Wikipedia results, run once with `--reset-continuation` or delete the matching file in `data/state/`.